# ***Task 2.1 — Dataset Selection and Setup***

**Paper:** Pegasos: Primal Estimated sub-GrAdient SOlver for SVM  

## *Dataset Choice: Breast Cancer Wisconsin (Diagnostic)*

I use the **Breast Cancer Wisconsin (Diagnostic)** dataset from `sklearn.datasets.load_breast_cancer()`. This dataset contains **569 samples** and **30 numeric features** extracted from digitised images of fine needle aspirate biopsies of breast masses. Each sample is labelled as **malignant (0)** or **benign (1)**, making it a binary classification problem — exactly the task that SVM (and Pegasos) is designed for.

**Why it is a reasonable testbed:** Pegasos solves the L2-regularised hinge-loss SVM for binary classification. The Breast Cancer dataset is a well-structured, fully numeric binary classification dataset with real-world geometric structure (the two classes partially overlap, creating a meaningful support vector boundary). The 30 features are scale-different, so preprocessing (StandardScaler + L2 normalisation) is needed — this lets me explicitly demonstrate and verify Assumption 1 from Task 1.2 (bounded feature norms `‖x‖ ≤ 1`).

**Limitations compared to the original paper's datasets:** The paper's primary experiments use large text datasets such as Reuters-21578 (23,149 samples, tens of thousands of sparse features) and the Covertype dataset (581,012 samples). In contrast, the Breast Cancer dataset is tiny (569 samples, 30 dense features), which means Pegasos's main advantage — sublinear-in-m convergence — is not stressed. On small datasets, a standard `sklearn` SVM solver would converge just as fast. This dataset therefore demonstrates **correctness of the algorithm**, not the scalability claim.


In [1]:
SEED = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(SEED)

The cell above imports all required libraries and fixes the global random seed to ensure reproducibility across runs.

In [2]:
data = load_breast_cancer()
X, y = data.data, data.target

y_svm = np.where(y == 0, -1, 1)

print(f"Dataset: {data.target_names}")
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Class balance: {np.bincount(y)} (0=malignant, 1=benign)")

Dataset: ['malignant' 'benign']
Samples: 569, Features: 30
Class balance: [212 357] (0=malignant, 1=benign)


We load the dataset and remap labels to {−1, +1} as required by the SVM hinge loss formulation `max(0, 1 − y⟨w, x⟩)`. This label convention is used throughout the paper (Section 2).

In [3]:
from sklearn.preprocessing import normalize

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_norm = normalize(X_scaled, norm='l2')

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_svm, test_size=0.2, random_state=SEED, stratify=y_svm
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Max feature norm in training set: {np.linalg.norm(X_train, axis=1).max():.4f}")

Train: (455, 30), Test: (114, 30)
Max feature norm in training set: 1.0000


***Preprocessing steps:***
1. **StandardScaler**: Centres each feature to zero mean and unit variance — removes scale differences across the 30 features.
2. **L2 row normalisation**: Projects each sample onto the unit sphere so that `‖x_i‖ = 1` for all i. This directly satisfies the paper's assumption `‖x‖ ≤ 1` (Theorem 1, Section 2), which is required for the convergence bound to hold.
3. **Stratified 80/20 split**: Preserves class balance in both train and test sets.
